In [73]:
###pip install jupyter python-docx


In [74]:
from pathlib import Path
import re
from collections import Counter
from docx import Document

base_path = Path(".")
master_resume = (base_path / "master_resume.txt").read_text(encoding="utf-8")
skills_list = [s.strip() for s in (base_path / "skills_dictionary.txt").read_text(encoding="utf-8").splitlines() if s.strip()]

## 3.2 Paste job description

In [75]:
job_description = """
Full job description
Posted date
Monday, May 5, 2025
Start date
As soon as possible
Job type
Contract
Work hours
Full-Time
Competition number
ATF2517
Salary
Commensurate with experience
Number of vacancies
1
Unit/Department
Division of Cardiology
Term
One-year contract with possibility to renew after the term.

Overview
The University of Ottawa Heart Institute (UOHI) is Canadas largest and foremost cardiac health centre, dedicated to advancing cardiovascular medicine through research, innovation, and patient-centred care. With over 1,500 staff members, including leading scientists and clinicians, UOHI is at the forefront of cardiac imaging, artificial intelligence, and minimally invasive interventions. Our institute combines cutting-edge technology with world-class expertise, pioneering new approaches in cardiac diagnosis, treatment, and prevention.

Under the leadership of Dr. Pascal Thériault-Lauzier, an interventional cardiologist and researcher in the Division of Cardiology, our research program integrates artificial intelligence, medical imaging, and interventional cardiology to enhance patient care and procedural outcomes. Our projects focus on automated cardiac image analysis, AI-assisted interventional planning, and predictive modelling for cardiovascular disease. We aim to develop innovative computational tools that support clinicians in making faster, more accurate decisions, ultimately improving patient outcomes.

We are seeking a Software Developer and AI Engineer to join our team. This role involves developing, training, and deploying AI models for automated medical image analysis, working closely with cardiologists and data scientists. The ideal candidate will be passionate about AI-driven healthcare solutions, contributing to state-of-the-art software development in a high-impact clinical setting. He or she is a natural problem solver with a strong foundation in data science.

Major areas of direct responsibility include:
Develop, train and deploy AI models that enable automated medical image analysis. Collaborate closely with the physicians to make the analysis of medical images more efficient and streamlined.
Assist in the design and development of software for medical devices.
Participate in code reviews and maintain standard coding practices.
Work closely with a team to integrate systems and manage troubleshooting.
Assist in the documentation of software development and testing.
Learn and implement new technologies under the guidance of senior staff.
Requirements:
Bachelor’s degree in software engineering, computer science, biomedical engineering, physics, mathematics or related field. A graduate degree would be an asset.
Up to 5 years of experience in software development.
Proficiency in web technology including Javascript, Typescript, and frameworks such as React or Angular.
Experience with AI models including convolutional neural networks and transformers for 3D image segmentation and 3D mesh generation.
Experience with python and artificial intelligence frameworks such as Pytorch, Tensorflow, or Keras.
Experience with cloud services such as Amazon Web Services and Google Cloud Platform would be an asset.
Knowledge of container technology such as Docker, Kubernetes or Apptainer.
Experience with cloud-based AI platforms such as VertexAI or SageMaker would be an asset.
Experience with executing computational work on High Performance Computing cluster environment would be an asset.
Experience with medical imaging data formats (DICOM, DICOMweb, NIfTI, FHIR, HL7) would be an asset.
Knowledge of a 3D graphics and compute API such as WebGL, OpenGL, Metal, Vulkan, CUDA, or WebGPU would be an asset.
Experience with container technology such as Docker, Kubernetes and Apptainer are assets.
Suitability:
Excellent interpersonal skills
Excellent organizational, analytical skills and decision-making skills
Good oral and written communications skills
Excellent analytical and time management skills;
High levels of creativity/innovation and adaptability to change;
Excellent teamwork and ability to transfer knowledge to other colleagues. """


In [76]:
import os
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

prompt = f"""
You are helping a junior Data/Business Intelligence Analyst write a resume.

Candidate base profile:
- Graduate of Artificial Intelligence Software Development and current Business Intelligence Systems student.
- Experience with Python, SQL, Power BI, Excel, pandas, NumPy, scikit-learn, RapidMiner, Jupyter Notebook, Azure, Git.
- Academic projects in predictive modelling, NLP, reinforcement learning, risk analysis, and Spotify data analysis.

Job description:
\"\"\"{job_description}\"\"\"

Candidate skills list:
{", ".join(skills_list)}

TASK:
1) Write a 3–4 sentence resume SUMMARY/Personal Profile targeting THIS job. Use natural language, mention 3–5 of the most relevant skills and responsibilities from the job description, and keep it in third person (no "I").
2) Create one concise SKILLS line, starting with the most relevant tools/skills for THIS job, then the rest, formatted exactly like:
Skills: Python, SQL, Power BI, ...

VERY IMPORTANT: reply in this exact format, nothing else:

SUMMARY:
<the summary here>

SKILLS:
Skills: Python, ...
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

content = response.text

# Simple parsing of the two sections
summary_part = ""
skills_part = ""

if "SUMMARY:" in content and "SKILLS:" in content:
    _, after_summary = content.split("SUMMARY:", 1)
    summary_part, after_summary = after_summary.split("SKILLS:", 1)
    summary_text = summary_part.strip()
    skills_text = after_summary.strip()
else:
    # Fallback: put everything in summary, and use a basic skills line
    summary_text = content.strip()
    skills_text = "Skills: " + ", ".join(skills_list)

print("SUMMARY TO USE:\n")
print(summary_text)
print("\nSKILLS BLOCK TO USE:\n")
print(skills_text)


SUMMARY TO USE:

A graduate of Artificial Intelligence Software Development, specializing in developing, training, and deploying AI models for predictive analytics. Possesses a strong foundation in Python, Azure, and software development, with academic experience in complex data analysis and machine learning. This candidate is prepared to contribute to AI-driven healthcare solutions, enhancing medical image analysis and clinical decision-making within a collaborative research setting.

SKILLS BLOCK TO USE:

Skills: Python, scikit-learn, Azure, Git, pandas, NumPy, Jupyter Notebook, Predictive Modelling, NLP, Reinforcement Learning, SQL, data cleaning, data visualization, statistics, Power BI, Excel, Tableau, RapidMiner, ETL, dashboards, risk analysis, governance, reporting, stakeholders


In [77]:
jd_lower = job_description.lower()

matched_skills = [s for s in skills_list if s.lower() in jd_lower]
matched_skills


['Python', 'Excel']

In [78]:
"""def match_skills(job_description, skills_list):
    jd_lower = job_description.lower()
    return [s for s in skills_list if s.lower() in jd_lower]

def build_summary_text(matched_skills, default_title="Junior Data / Business Intelligence Analyst"):
    # Take up to 4 key skills from the description
    top = ", ".join(matched_skills[:4]) if matched_skills else "Python, SQL, Power BI"
    summary = (
        f"{default_title} with graduate training in Artificial Intelligence Software Development and Business Intelligence Systems, "
        f"experienced in {top} for data analysis and reporting. "
        "Strong foundation in machine learning, data cleaning, and statistical analysis, with academic projects in predictive modelling, NLP, and reinforcement learning. "
        "Able to translate complex data into clear insights and dashboards, and comfortable working in fast-paced, team-based environments."
    )
    return summary

def build_skills_block(matched_skills, skills_list):

    Creates a short Skills section text, emphasizing matched skills first.
    
    ordered = matched_skills + [s for s in skills_list if s not in matched_skills]
    # Simple comma-separated list; you can later format into bullet lines manually if you want.
    return "Skills: " + ", ".join(ordered)

    """


'def match_skills(job_description, skills_list):\n    jd_lower = job_description.lower()\n    return [s for s in skills_list if s.lower() in jd_lower]\n\ndef build_summary_text(matched_skills, default_title="Junior Data / Business Intelligence Analyst"):\n    # Take up to 4 key skills from the description\n    top = ", ".join(matched_skills[:4]) if matched_skills else "Python, SQL, Power BI"\n    summary = (\n        f"{default_title} with graduate training in Artificial Intelligence Software Development and Business Intelligence Systems, "\n        f"experienced in {top} for data analysis and reporting. "\n        "Strong foundation in machine learning, data cleaning, and statistical analysis, with academic projects in predictive modelling, NLP, and reinforcement learning. "\n        "Able to translate complex data into clear insights and dashboards, and comfortable working in fast-paced, team-based environments."\n    )\n    return summary\n\ndef build_skills_block(matched_skills, sk

API here

In [79]:
import json

# Optional: still compute matched_skills via string match
jd_lower = job_description.lower()
matched_skills = [s for s in skills_list if s.lower() in jd_lower]

print("Matched skills (simple):", matched_skills)


Matched skills (simple): ['Python', 'Excel']


In [80]:
"""matched_skills = match_skills(job_description, skills_list)

# You can change the title depending on the job: "Data Analyst" or "Business Intelligence Analyst"
summary_text = build_summary_text(matched_skills, default_title="Data Analyst")
skills_text = build_skills_block(matched_skills, skills_list)

print("Matched skills:", matched_skills)
print("\nSUMMARY TO USE:\n")
print(summary_text)
print("\nSKILLS BLOCK TO USE:\n")
print(skills_text)"""


'matched_skills = match_skills(job_description, skills_list)\n\n# You can change the title depending on the job: "Data Analyst" or "Business Intelligence Analyst"\nsummary_text = build_summary_text(matched_skills, default_title="Data Analyst")\nskills_text = build_skills_block(matched_skills, skills_list)\n\nprint("Matched skills:", matched_skills)\nprint("\nSUMMARY TO USE:\n")\nprint(summary_text)\nprint("\nSKILLS BLOCK TO USE:\n")\nprint(skills_text)'

In [81]:
output_dir = base_path / "output"
output_dir.mkdir(exist_ok=True)

base_path = Path(".")
output_dir = base_path / "output"
output_dir.mkdir(exist_ok=True)

template_path = base_path / "resume_template.docx"
doc = Document(template_path)

def replace_placeholder(doc, placeholder, new_text):
    # Replace in paragraphs
    for p in doc.paragraphs:
        if placeholder in p.text:
            p.text = p.text.replace(placeholder, new_text)
    # Replace in tables (in case some sections are in tables)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if placeholder in cell.text:
                    cell.text = cell.text.replace(placeholder, new_text)

# Replace SUMMARY and SKILLS only
replace_placeholder(doc, "{{SUMMARY}}", summary_text)
replace_placeholder(doc, "{{SKILLS}}", skills_text)

file_name = "Adhithya_Rajesh_Tailored_Resume.docx"
file_path = output_dir / file_name
doc.save(file_path)
file_path



WindowsPath('output/Adhithya_Rajesh_Tailored_Resume.docx')

In [82]:
###pip install google-genai
